# Bài thực hành 4: So sánh ảnh bằng Wavelet Hash

**Chương 3 – Phần 1: Phát hiện đặc trưng và đối sánh ảnh**

Notebook được tổ chức đúng **6 phần của đề bài**:

1. Chuẩn bị dữ liệu.
2. Trích xuất ma trận Wavelet.
3. Tạo mã băm từ hệ số Wavelet đã lượng tử hóa.
4. So sánh mã băm bằng khoảng cách Hamming.
5. Đánh giá Accuracy, Sensitivity, Specificity và ROC/AUC.
6. Khảo sát các phương pháp Wavelet Hash khác nhau.

Hàm nào phục vụ phần nào được viết ngay trong phần đó. Hãy chạy lần lượt bằng **Run All**.


## 1. Chuẩn bị dữ liệu

Chuẩn bị hai nhóm ảnh:

```text
image/
├── tuong_tu/          # Ít nhất 3 ảnh cùng đối tượng/cảnh
└── khong_tuong_tu/    # Ít nhất 1 ảnh khác đối tượng/cảnh
```

Ví dụ nhóm tương tự gồm cùng một vật ở các góc chụp, độ sáng hoặc mức nhiễu khác nhau. Nhóm không tương tự phải chứa đối tượng/cảnh khác.

**Input:** ảnh `.jpg`, `.jpeg`, `.png`, `.bmp`, `.tif`, `.tiff` hoặc `.webp` trong hai thư mục trên.

**Output:** danh sách ảnh RGB, nhãn nhóm và hình kiểm tra dữ liệu. Tiêu đề xanh là tương tự; đỏ là không tương tự.


In [ ]:
# ============================================================
# PHẦN 1: ĐỌC VÀ KIỂM TRA DỮ LIỆU
# Input : image/tuong_tu và image/khong_tuong_tu
# Output: images, similar_paths, dissimilar_paths
# ============================================================
from pathlib import Path
import itertools
import subprocess
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np

# PyWavelets có tên gói cài đặt là PyWavelets nhưng tên import là pywt.
# Nếu kernel Jupyter hiện tại chưa có, đoạn này cài vào đúng kernel đang chạy.
try:
    import pywt
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "PyWavelets"])
    import pywt

# Xác định thư mục bài dù notebook được mở từ thư mục gốc hay PHAN_1.
ROOT = Path.cwd()
if not (ROOT / "image").exists():
    candidate = ROOT / "CHUONG_3" / "PHAN_1"
    if candidate.exists():
        ROOT = candidate

SIMILAR_DIR = ROOT / "image" / "tuong_tu"
DISSIMILAR_DIR = ROOT / "image" / "khong_tuong_tu"
OUTPUT_DIR = ROOT / "outputs"
SIMILAR_DIR.mkdir(parents=True, exist_ok=True)
DISSIMILAR_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
RNG = np.random.default_rng(42)  # Seed cố định để kết quả chia tập có thể lặp lại.
plt.rcParams.update({"figure.figsize": (12, 6), "axes.titlesize": 10})


In [ ]:
# BƯỚC NHỎ 1.2: Khai báo hàm tìm và đọc ảnh
def list_images(folder):
    """Lấy các file ảnh hợp lệ trong thư mục và sắp xếp theo tên."""
    return sorted(
        path for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in EXTENSIONS
    )


def read_rgb(path):
    """Đọc ảnh bằng OpenCV rồi đổi BGR/BGRA/Gray sang RGB/RGBA."""
    image = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if image is None:
        raise ValueError(f"Không đọc được ảnh: {path}")
    if image.ndim == 2:
        return cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    if image.shape[2] == 4:
        return cv2.cvtColor(image, cv2.COLOR_BGRA2RGBA)
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


In [ ]:
# BƯỚC NHỎ 1.3: Đọc dữ liệu từ hai thư mục và kiểm tra số lượng
similar_paths = list_images(SIMILAR_DIR)
dissimilar_paths = list_images(DISSIMILAR_DIR)

# Cần ít nhất 3 ảnh tương tự để tạo >= 3 cặp dương và chia validation/test.
if len(similar_paths) < 3 or len(dissimilar_paths) < 1:
    raise ValueError(
        "Thiếu dữ liệu: hãy đặt ít nhất 3 ảnh vào image/tuong_tu "
        "và ít nhất 1 ảnh vào image/khong_tuong_tu."
    )

images = {path.name: read_rgb(path) for path in similar_paths + dissimilar_paths}
group_labels = {
    **{path.name: 1 for path in similar_paths},
    **{path.name: 0 for path in dissimilar_paths},
}

print(f"[MÔI TRƯỜNG] Python: {sys.executable}")
print(f"[MÔI TRƯỜNG] PyWavelets: {pywt.__version__}")
print(f"[OUTPUT] Đọc được {len(similar_paths)} ảnh tương tự và {len(dissimilar_paths)} ảnh không tương tự.")


In [ ]:
# BƯỚC NHỎ 1.4: Hiển thị ảnh để kiểm tra nhãn bằng mắt
# Hiển thị dữ liệu để phát hiện sớm việc bỏ ảnh nhầm thư mục.
cols = min(4, len(images))
rows = int(np.ceil(len(images) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.4 * rows), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for ax, (name, image) in zip(axes.flat, images.items()):
    is_similar = group_labels[name] == 1
    ax.imshow(image)
    ax.set_title(
        ("TƯƠNG TỰ\n" if is_similar else "KHÔNG TƯƠNG TỰ\n") + name,
        color="green" if is_similar else "red",
    )
    ax.axis("off")
plt.tight_layout()
print("[NHẬN XÉT] Hãy kiểm tra bằng mắt xem từng ảnh đã nằm đúng nhóm chưa.")


## 2. Trích xuất ma trận Wavelet

Biến đổi Wavelet 2D phân rã ảnh thành:

- `LL`: ma trận xấp xỉ, giữ cấu trúc tổng quát.
- `LH`, `HL`, `HH`: các ma trận chi tiết theo hướng.

**Input:** toàn bộ ảnh RGB của phần 1.

**Output:** ma trận Wavelet của từng ảnh và hình minh họa các dải Wavelet của ảnh đầu tiên.


In [ ]:
# ============================================================
# PHẦN 2: ẢNH -> CÁC MA TRẬN WAVELET
# Input : images
# Output: wavelet_matrices chứa kết quả phân rã của từng ảnh
# ============================================================
def to_gray(image, image_size):
    """Đổi ảnh sang ảnh xám vuông và chuẩn hóa giá trị về [0, 1]."""
    data = np.asarray(image)
    if data.ndim == 3:
        conversion = cv2.COLOR_RGBA2GRAY if data.shape[2] == 4 else cv2.COLOR_RGB2GRAY
        data = cv2.cvtColor(data, conversion)
    elif data.ndim != 2:
        raise ValueError("Ảnh phải có dạng HxW, HxWx3 hoặc HxWx4")

    data = cv2.resize(data, (image_size, image_size), interpolation=cv2.INTER_AREA)
    data = data.astype(np.float32)
    if data.max() > 1.0:
        data /= 255.0
    return np.clip(data, 0.0, 1.0)


def extract_wavelet_matrices(image, wavelet="db4", level=2, image_size=256):
    """Trả về các ma trận Wavelet đa mức do pywt.wavedec2 tạo ra."""
    gray = to_gray(image, image_size)
    coefficients = pywt.wavedec2(gray, wavelet=wavelet, level=level)
    # coefficients[0] là LL ở mức cuối; phần sau là (LH, HL, HH) từng mức.
    return gray, coefficients


In [ ]:
# BƯỚC NHỎ 2.2: Trích xuất ma trận Wavelet cho tất cả ảnh
# Trích xuất cho TỪNG ảnh đúng yêu cầu đề bài.
wavelet_matrices = {
    name: extract_wavelet_matrices(image, wavelet="db4", level=2)
    for name, image in images.items()
}

print("[OUTPUT] Kích thước ma trận LL mức cuối của từng ảnh:")
for name, (_, coefficients) in wavelet_matrices.items():
    approximation_ll = coefficients[0]
    print(f"  {name:35s} -> LL shape = {approximation_ll.shape}")


In [ ]:
# BƯỚC NHỎ 2.3: Hiển thị trực quan LL, LH, HL, HH
# Minh họa trực quan phân rã mức 1 trên ảnh đầu tiên.
sample_name = similar_paths[0].name
sample_gray = to_gray(images[sample_name], 256)
ll, (lh, hl, hh) = pywt.dwt2(sample_gray, "db4")
bands = [sample_gray, ll, lh, hl, hh]
titles = ["Ảnh xám", "LL – xấp xỉ", "LH – chi tiết", "HL – chi tiết", "HH – chi tiết"]

fig, axes = plt.subplots(1, 5, figsize=(16, 4))
for ax, matrix, title in zip(axes, bands, titles):
    ax.imshow(matrix, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
print(f"[NHẬN XÉT] Ảnh minh họa: {sample_name}. LL giữ hình dáng tổng quát rõ nhất.")


## 3. Tạo mã băm từ hệ số Wavelet đã lượng tử hóa

Mỗi ảnh được phân rã Wavelet, lấy ma trận `LL`, thu về kích thước `8 × 8`, rồi lượng tử hóa quanh giá trị trung vị:

- Hệ số lớn hơn trung vị → bit `1`.
- Hệ số nhỏ hơn hoặc bằng trung vị → bit `0`.

**Input:** ảnh và các tham số `wavelet="db4"`, `level=2`, `hash_size=8`.

**Output:** một mã Wavelet Hash 64 bit cho mỗi ảnh, được in dạng hexadecimal cho gọn.


In [ ]:
# ============================================================
# PHẦN 3: MA TRẬN WAVELET LL -> LƯỢNG TỬ HÓA -> HASH 64 BIT
# Input : images
# Output: hashes
# ============================================================
def wavelet_hash(image, wavelet="db4", level=2, hash_size=8):
    """Tạo mã Wavelet Hash nhị phân có đúng hash_size² bit."""
    if hash_size < 2 or level < 1:
        raise ValueError("hash_size phải >= 2 và level phải >= 1")

    # Ảnh đầu vào 32x32 khi hash_size=8, level=2.
    image_size = hash_size * (2 ** level)
    gray = to_gray(image, image_size)
    max_level = pywt.dwtn_max_level(gray.shape, wavelet)
    if level > max_level:
        raise ValueError(f"level={level} quá lớn; mức tối đa là {max_level}")

    # Chỉ lấy LL vì thành phần tần số thấp ổn định hơn trước nhiễu nhỏ.
    approximation_ll = pywt.wavedec2(gray, wavelet=wavelet, level=level)[0]
    ll_8x8 = cv2.resize(
        approximation_ll, (hash_size, hash_size), interpolation=cv2.INTER_AREA
    )

    # Lượng tử hóa ma trận số thực thành ma trận bit rồi trải thành 64 bit.
    quantization_threshold = np.median(ll_8x8)
    return (ll_8x8 > quantization_threshold).reshape(-1)


In [ ]:
# BƯỚC NHỎ 3.2: Đổi mã bit sang hexadecimal để dễ đọc
def hash_to_hex(hash_code):
    """Đổi mã bit sang hexadecimal để dễ đọc và lưu trên console."""
    bits = np.asarray(hash_code, dtype=np.uint8).reshape(-1)
    if (-bits.size) % 4:
        bits = np.pad(bits, (0, (-bits.size) % 4))
    return "".join(
        f"{int(''.join(map(str, nibble)), 2):x}"
        for nibble in bits.reshape(-1, 4)
    )


In [ ]:
# BƯỚC NHỎ 3.3: Tạo và in mã băm cho từng ảnh
hashes = {name: wavelet_hash(image) for name, image in images.items()}

print("[OUTPUT] Wavelet Hash của từng ảnh:")
for name, hash_code in hashes.items():
    print(f"  {name:35s} | {hash_code.size} bit | {hash_to_hex(hash_code)}")
print("[NHẬN XÉT] Mọi ảnh đều phải tạo đúng 64 bit để có thể so sánh với nhau.")


## 4. So sánh mã băm bằng khoảng cách Hamming

Khoảng cách Hamming là số vị trí bit khác nhau giữa hai mã băm:

- Khoảng cách nhỏ → hai ảnh có xu hướng tương tự.
- Khoảng cách lớn → hai ảnh có xu hướng không tương tự.

**Input:** các mã băm 64 bit của phần 3.

**Output:** danh sách cặp ảnh, nhãn thật và khoảng cách Hamming. Nhãn `1` là tương tự; nhãn `0` là không tương tự.


In [ ]:
# ============================================================
# PHẦN 4: SO SÁNH HAI HASH BẰNG HAMMING
# Input : hashes
# Output: records = (ảnh trái, ảnh phải, nhãn thật, Hamming)
# ============================================================
def hamming_distance(hash1, hash2):
    """Đếm số bit khác nhau giữa hai mã băm cùng độ dài."""
    a = np.asarray(hash1, dtype=bool).reshape(-1)
    b = np.asarray(hash2, dtype=bool).reshape(-1)
    if a.shape != b.shape:
        raise ValueError("Hai mã băm phải có cùng độ dài")
    return int(np.count_nonzero(a != b))


In [ ]:
# BƯỚC NHỎ 4.2: Tạo cặp tương tự và cặp không tương tự
similar_names = [path.name for path in similar_paths]
dissimilar_names = [path.name for path in dissimilar_paths]
records = []

# Các cặp nằm hoàn toàn trong tuong_tu là cặp dương, nhãn 1.
for left, right in itertools.combinations(similar_names, 2):
    distance = hamming_distance(hashes[left], hashes[right])
    records.append((left, right, 1, distance))

# Mỗi ảnh tương tự ghép với mỗi ảnh không tương tự tạo cặp âm, nhãn 0.
for left, right in itertools.product(similar_names, dissimilar_names):
    distance = hamming_distance(hashes[left], hashes[right])
    records.append((left, right, 0, distance))


In [ ]:
# BƯỚC NHỎ 4.3: In và so sánh khoảng cách của hai nhóm
print("[OUTPUT] Khoảng cách Hamming của từng cặp:")
for left, right, label, distance in records:
    group = "tương tự" if label == 1 else "không tương tự"
    print(f"  nhãn={label} ({group:16s}) | Hamming={distance:2d} | {left} <-> {right}")

positive_distances = [record[3] for record in records if record[2] == 1]
negative_distances = [record[3] for record in records if record[2] == 0]
print(f"[NHẬN XÉT] Hamming trung bình cặp tương tự: {np.mean(positive_distances):.2f}")
print(f"[NHẬN XÉT] Hamming trung bình cặp không tương tự: {np.mean(negative_distances):.2f}")


## 5. Đánh giá thuật toán

Tập cặp ảnh được chia theo từng nhãn:

- **Validation:** chọn ngưỡng Hamming.
- **Test:** đánh giá cuối, không dùng để chọn lại ngưỡng.

Quy tắc: `Hamming <= ngưỡng` thì dự đoán hai ảnh tương tự.

**Output:** Accuracy, Sensitivity, Specificity, Precision, confusion matrix, histogram khoảng cách, ROC và AUC.


In [ ]:
# ============================================================
# PHẦN 5: CHỌN NGƯỠNG VÀ ĐÁNH GIÁ
# Input : records
# Output: metrics, ROC/AUC và biểu đồ đánh giá
# ============================================================
def classification_metrics(labels, predictions):
    """Tính confusion matrix và các chỉ số phân loại."""
    y = np.asarray(labels, dtype=int)       # Nhãn thật
    p = np.asarray(predictions, dtype=int)  # Nhãn dự đoán

    tp = int(np.sum((y == 1) & (p == 1)))  # Tương tự, dự đoán tương tự
    tn = int(np.sum((y == 0) & (p == 0)))  # Không tương tự, dự đoán không tương tự
    fp = int(np.sum((y == 0) & (p == 1)))  # Không tương tự nhưng dự đoán tương tự
    fn = int(np.sum((y == 1) & (p == 0)))  # Tương tự nhưng dự đoán không tương tự
    safe = lambda numerator, denominator: numerator / denominator if denominator else 0.0

    return {
        "accuracy": safe(tp + tn, len(y)),  # Tỷ lệ tất cả cặp phân loại đúng
        "sensitivity": safe(tp, tp + fn),   # Tỷ lệ cặp tương tự nhận đúng
        "specificity": safe(tn, tn + fp),   # Tỷ lệ cặp không tương tự nhận đúng
        "precision": safe(tp, tp + fp),
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
    }


def roc_curve_from_distances(labels, distances):
    """Quét toàn bộ ngưỡng Hamming để tạo ROC và tính AUC."""
    y = np.asarray(labels, dtype=int)
    d = np.asarray(distances, dtype=float)
    thresholds = np.r_[-np.inf, np.unique(d), np.inf]
    points = []
    for threshold in thresholds:
        metric = classification_metrics(y, d <= threshold)
        fpr = 1.0 - metric["specificity"]
        tpr = metric["sensitivity"]
        points.append((fpr, tpr, threshold))

    points.sort(key=lambda item: (item[0], item[1]))
    fpr, tpr, ordered_thresholds = map(np.asarray, zip(*points))
    auc = float(np.trapezoid(tpr, fpr))
    return fpr, tpr, ordered_thresholds, auc


In [ ]:
# BƯỚC NHỎ 5.2: Chia các cặp thành validation và test
# Chia riêng từng lớp để validation/test đều có cặp tương tự và không tương tự.
positive = [record for record in records if record[2] == 1]
negative = [record for record in records if record[2] == 0]
if len(positive) < 2 or len(negative) < 2:
    raise ValueError("Cần ít nhất 2 cặp mỗi lớp để chia validation/test.")
RNG.shuffle(positive)
RNG.shuffle(negative)


def split_class(items):
    """Chia đôi nhưng bảo đảm mỗi nửa có ít nhất một phần tử."""
    cut = max(1, min(len(items) - 1, len(items) // 2))
    return items[:cut], items[cut:]


val_pos, test_pos = split_class(positive)
val_neg, test_neg = split_class(negative)
validation = val_pos + val_neg
test = test_pos + test_neg


In [ ]:
# BƯỚC NHỎ 5.3: Chọn ngưỡng trên validation và đánh giá test
# Chọn ngưỡng tối đa hóa balanced accuracy trên validation.
val_labels = np.array([record[2] for record in validation])
val_distances = np.array([record[3] for record in validation])
candidate_thresholds = np.arange(65)  # Hash 64 bit nên Hamming nằm trong 0..64.
validation_scores = []
for threshold in candidate_thresholds:
    metric_at_threshold = classification_metrics(val_labels, val_distances <= threshold)
    balanced_accuracy = (
        metric_at_threshold["sensitivity"] + metric_at_threshold["specificity"]
    ) / 2
    validation_scores.append(balanced_accuracy)
best_threshold = int(candidate_thresholds[np.argmax(validation_scores)])

# Chỉ dùng ngưỡng đã chọn để dự đoán tập test.
test_labels = np.array([record[2] for record in test])
test_distances = np.array([record[3] for record in test])
test_predictions = (test_distances <= best_threshold).astype(int)
metric = classification_metrics(test_labels, test_predictions)
fpr, tpr, _, auc = roc_curve_from_distances(test_labels, test_distances)

print(f"[OUTPUT] Validation: {len(validation)} cặp | Test: {len(test)} cặp")
print(f"[OUTPUT] Ngưỡng Hamming được chọn: {best_threshold}")
print(f"[OUTPUT] Accuracy    : {metric['accuracy']:.3f}")
print(f"[OUTPUT] Sensitivity : {metric['sensitivity']:.3f}")
print(f"[OUTPUT] Specificity : {metric['specificity']:.3f}")
print(f"[OUTPUT] Precision   : {metric['precision']:.3f}")
print(f"[OUTPUT] AUC         : {auc:.3f}")
print(f"[OUTPUT] TP={metric['tp']}, TN={metric['tn']}, FP={metric['fp']}, FN={metric['fn']}")


In [ ]:
# BƯỚC NHỎ 5.4: Vẽ histogram, ROC và ma trận nhầm lẫn
# Biểu đồ 1: histogram; biểu đồ 2: ROC; biểu đồ 3: confusion matrix.
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].hist(test_distances[test_labels == 1], bins=np.arange(0, 66, 2), alpha=.7, label="Tương tự")
axes[0].hist(test_distances[test_labels == 0], bins=np.arange(0, 66, 2), alpha=.7, label="Không tương tự")
axes[0].axvline(best_threshold, color="black", linestyle="--", label=f"ngưỡng={best_threshold}")
axes[0].set(xlabel="Khoảng cách Hamming", ylabel="Số cặp", title="Phân bố khoảng cách")
axes[0].legend()

axes[1].plot(fpr, tpr, marker=".", label=f"AUC={auc:.3f}")
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set(xlabel="False Positive Rate", ylabel="True Positive Rate", title="Đường cong ROC",
            xlim=(0, 1), ylim=(0, 1.02))
axes[1].grid(alpha=.3)
axes[1].legend()

confusion_matrix = np.array([[metric["tn"], metric["fp"]], [metric["fn"], metric["tp"]]])
axes[2].imshow(confusion_matrix, cmap="Blues")
for row in range(2):
    for col in range(2):
        axes[2].text(col, row, confusion_matrix[row, col], ha="center", va="center", fontsize=14)
axes[2].set(
    xticks=[0, 1], yticks=[0, 1],
    xticklabels=["Không tương tự", "Tương tự"],
    yticklabels=["Không tương tự", "Tương tự"],
    xlabel="Dự đoán", ylabel="Thực tế", title="Ma trận nhầm lẫn",
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "wavelet_hash_evaluation.png", dpi=150, bbox_inches="tight")
print("[NHẬN XÉT] Kết quả tốt khi hai histogram tách nhau, AUC gần 1 và FP/FN thấp.")


## 6. Bài tập nâng cao: khảo sát các phương pháp Wavelet Hash

So sánh bốn họ Wavelet: `haar`, `db2`, `db4`, `sym4`.

Để so sánh công bằng, mỗi phương pháp:

1. Tạo lại hash cho cùng tập ảnh.
2. Chọn ngưỡng riêng trên cùng tập validation.
3. Đánh giá trên cùng tập test.

**Output:** ngưỡng, Accuracy, Sensitivity, Specificity, AUC và biểu đồ so sánh hiệu suất của từng phương pháp.


In [ ]:
# ============================================================
# PHẦN 6: SO SÁNH CÁC HỌ WAVELET
# Input : cùng validation/test, thay đổi họ wavelet
# Output: bảng chỉ số và biểu đồ Accuracy/AUC
# ============================================================
wavelet_families = ["haar", "db2", "db4", "sym4"]
comparison = []

for family in wavelet_families:
    # Bước 1: tạo lại hash cho mọi ảnh bằng họ Wavelet đang khảo sát.
    family_hashes = {
        name: wavelet_hash(image, wavelet=family, level=2, hash_size=8)
        for name, image in images.items()
    }

    # Bước 2: tính Hamming và chọn ngưỡng riêng trên validation.
    family_val_distances = np.array([
        hamming_distance(family_hashes[left], family_hashes[right])
        for left, right, _, _ in validation
    ])
    family_validation_scores = []
    for threshold in candidate_thresholds:
        family_metric = classification_metrics(val_labels, family_val_distances <= threshold)
        score = (family_metric["sensitivity"] + family_metric["specificity"]) / 2
        family_validation_scores.append(score)
    family_threshold = int(candidate_thresholds[np.argmax(family_validation_scores)])

    # Bước 3: đánh giá phương pháp trên cùng tập test.
    family_test_distances = np.array([
        hamming_distance(family_hashes[left], family_hashes[right])
        for left, right, _, _ in test
    ])
    family_predictions = family_test_distances <= family_threshold
    family_metric = classification_metrics(test_labels, family_predictions)
    _, _, _, family_auc = roc_curve_from_distances(test_labels, family_test_distances)

    comparison.append({
        "wavelet": family,
        "threshold": family_threshold,
        "accuracy": family_metric["accuracy"],
        "sensitivity": family_metric["sensitivity"],
        "specificity": family_metric["specificity"],
        "auc": family_auc,
    })


In [ ]:
# BƯỚC NHỎ 6.2: In bảng kết quả của từng họ Wavelet
print("[OUTPUT] So sánh trên cùng tập test:")
for result in comparison:
    print(
        f"  {result['wavelet']:5s} | ngưỡng={result['threshold']:2d} "
        f"| Accuracy={result['accuracy']:.3f} "
        f"| Sensitivity={result['sensitivity']:.3f} "
        f"| Specificity={result['specificity']:.3f} "
        f"| AUC={result['auc']:.3f}"
    )


In [ ]:
# BƯỚC NHỎ 6.3: Vẽ biểu đồ và xác định phương pháp tốt nhất
# Vẽ Accuracy và AUC trên cùng thang 0..1 để dễ đối chiếu.
names = [result["wavelet"] for result in comparison]
accuracies = [result["accuracy"] for result in comparison]
aucs = [result["auc"] for result in comparison]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(names, accuracies, color="#4c78a8")
axes[0].set(title="So sánh Accuracy", ylabel="Accuracy", ylim=(0, 1.05))
axes[0].grid(axis="y", alpha=.3)
axes[1].bar(names, aucs, color="#f58518")
axes[1].set(title="So sánh AUC", ylabel="AUC", ylim=(0, 1.05))
axes[1].grid(axis="y", alpha=.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "wavelet_methods_comparison.png", dpi=150, bbox_inches="tight")

best_method = max(comparison, key=lambda result: (result["auc"], result["accuracy"]))
print(
    f"[NHẬN XÉT] Với tập ảnh hiện tại, {best_method['wavelet']} tốt nhất "
    f"theo ưu tiên AUC rồi Accuracy."
)
print("[LƯU Ý] Kết luận có thể thay đổi khi bộ ảnh hoặc mức biến đổi ảnh thay đổi.")
